[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SheffieldML/notebook/blob/main/GPc/gplvm_oil100.ipynb)

# MAP GPLVM on oil (100) with `gpc`

GPc port of GPy `gplvm_oil_100` / native [gplvm_oil100](https://github.com/SheffieldML/GPc/blob/main/docs/tutorials/gplvm_oil100.md).

Requires `gpc>=0.1.0a8` (CIP-001D / CIP-0039 façade; also on `0.1.0a7`).

```python
m = gpc.models.GPLVM(Y, input_dim=2)  # default RBF + Bias + White
m.optimize()
X = m.latent_mean()
```

Data via [`pods`](https://pypi.org/project/pods/) `oil_100` (labels colour the scatter; they are not supervised targets).


In [ ]:
try:
    import gpc
except ImportError:
    import subprocess
    import sys
    # Prefer PyPI; fall back to TestPyPI for pre-release alphas (GPc CIP-002C).
    cmd = [sys.executable, "-m", "pip", "install", "-q", "gpc"]
    try:
        subprocess.check_call(cmd)
    except subprocess.CalledProcessError:
        subprocess.check_call(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "--no-cache-dir",
                "--index-url",
                "https://test.pypi.org/simple/",
                "--extra-index-url",
                "https://pypi.org/simple/",
                "gpc>=0.1.0a8",
            ]
        )
    import gpc


In [ ]:
import os
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")

import numpy as np
import gpc
from matplotlib import pyplot as plt

try:
    import pods
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pods"])
    import pods

try:
    import matplotlib
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "matplotlib"])


## Load oil flow (100 points)

In [ ]:
data = pods.datasets.oil_100()
Y = np.asarray(data["X"], dtype=float)  # pods: features in X, class indicators in Y
labels = np.asarray(data["Y"], dtype=float)
print("Y obs", Y.shape, "labels", labels.shape)
# one-hot ±1 → class index 0..2
cls = np.argmax(labels, axis=1)

## Fit MAP GPLVM

In [ ]:
m = gpc.models.GPLVM(Y, input_dim=2)
print(m)
m.optimize(iters=80, messages=False)
print("objective", m.objective())
Xlat = m.latent_mean()
print("latent", Xlat.shape)

## Latent scatter coloured by oil class

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
for c, name in enumerate(("non-flowing", "slugging", "homogeneous")):
    mask = cls == c
    ax.scatter(Xlat[mask, 0], Xlat[mask, 1], s=28, label=name, alpha=0.85)
ax.set_xlabel("latent 0")
ax.set_ylabel("latent 1")
ax.set_title("MAP GPLVM — oil_100")
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

## Reconstruct a few points

In [ ]:
mu, var = m.predict(Xlat[:5])
print("predict mu", mu.shape, "var", var.shape)
print("RMSE (train, first 5):", float(np.sqrt(np.mean((mu - Y[:5]) ** 2))))